# 08.9 - Fine-tuning Transformers

**Phase:** 08 - Transformers

**Status:** VERIFIED

---

## 1. What Are We Solving?

Fine-tuning is the process of taking a pre-trained transformer and continuing training on task-specific data to adapt it to your use case.

## 2. Why Does This Matter?

Pre-trained models have general language understanding. Fine-tuning specializes that understanding for your specific task with far less data than training from scratch.

## 3. Prerequisites

- Hugging Face ecosystem (08.8), understanding of BERT/GPT (08.5, 08.6), training basics (Phase 06).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain when to fine-tune vs use a model as-is.
- Fine-tune a transformer for classification.
- Understand learning rate, freezing, and LoRA trade-offs.
- Evaluate fine-tuned models properly.

## 5. Mental Model

Fine-tuning is like hiring a well-educated generalist and training them for a specific job. The pre-trained model already knows language. Fine-tuning teaches it the task's specific patterns with task-specific data.

> NOTE: This notebook uses a tiny randomly-initialized transformer to demonstrate the fine-tuning workflow without downloading large pretrained weights.


## 6. Setup: Tiny Model & Synthetic Data

Create a tiny BERT-style model and a small synthetic classification dataset.


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
import numpy as np
from transformers import BertConfig, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

torch.manual_seed(42)
np.random.seed(42)

# Tiny model
config = BertConfig(
    vocab_size=100,
    hidden_size=32,
    num_hidden_layers=2,
    num_attention_heads=2,
    intermediate_size=64,
)
model = BertForSequenceClassification(BertConfig(**{**config.to_dict(), "num_labels": 2}))
print("Tiny model params:", sum(p.numel() for p in model.parameters()))

# Synthetic data: sequences of token ids with labels
def make_data(n):
    texts, labels = [], []
    for i in range(n):
        # Class 0: tokens 5-9, Class 1: tokens 10-14 (separable)
        if i % 2 == 0:
            ids = [5, 6, 7, 8, 9]
            label = 0
        else:
            ids = [10, 11, 12, 13, 14]
            label = 1
        texts.append(ids)
        labels.append(label)
    return texts, labels

train_ids, train_labels = make_data(40)
eval_ids, eval_labels = make_data(10)

train_ds = Dataset.from_dict({"input_ids": train_ids, "labels": train_labels})
eval_ds = Dataset.from_dict({"input_ids": eval_ids, "labels": eval_labels})
print("Train dataset size:", len(train_ds))
print("Eval dataset size:", len(eval_ds))


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tiny model params: 37922
Train dataset size: 40
Eval dataset size: 10


## 7. Fine-tune with the Trainer API

Use a small learning rate to preserve pre-trained knowledge.


In [2]:
args = TrainingArguments(
    output_dir="./finetune_results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,   # small LR for fine-tuning
    weight_decay=0.01,
    logging_steps=1,
    report_to=[],
    disable_tqdm=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print("\nFine-tuning complete.")


D:\CODE\complete ml\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': '0.6973', 'grad_norm': '0.4497', 'learning_rate': '0.0002', 'epoch': '0.2'}
{'loss': '0.6973', 'grad_norm': '0.3484', 'learning_rate': '0.0001867', 'epoch': '0.4'}
{'loss': '0.6985', 'grad_norm': '0.8234', 'learning_rate': '0.0001733', 'epoch': '0.6'}
{'loss': '0.6937', 'grad_norm': '0.4524', 'learning_rate': '0.00016', 'epoch': '0.8'}
{'loss': '0.6903', 'grad_norm': '0.4025', 'learning_rate': '0.0001467', 'epoch': '1'}


{'eval_loss': '0.6916', 'eval_accuracy': '0.5', 'eval_runtime': '0.3423', 'eval_samples_per_second': '29.22', 'eval_steps_per_second': '5.843', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 64.75it/s]

...

{'loss': '0.6894', 'grad_norm': '0.5855', 'learning_rate': '6.667e-05', 'epoch': '2.2'}
{'loss': '0.6881', 'grad_norm': '0.4488', 'learning_rate': '5.333e-05', 'epoch': '2.4'}
{'loss': '0.6868', 'grad_norm': '0.5763', 'learning_rate': '4e-05', 'epoch': '2.6'}
{'loss': '0.69', 'grad_norm': '0.3533', 'learning_rate': '2.667e-05', 'epoch': '2.8'}
{'loss': '0.6881', 'grad_norm': '0.4226', 'learning_rate': '1.333e-05', 'epoch': '3'}
{'eval_loss': '0.6875', 'eval_accuracy': '1', 'eval_runtime': '0.0151', 'eval_samples_per_second': '662', 'eval_steps_per_second': '132.4', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 74.39it/s]

{'train_runtime': '1.762', 'train_samples_per_second': '68.11', 'train_steps_per_second': '8.514', 'train_loss': '0.6912', 'epoch': '3'}



Fine-tuning complete.


## 8. Evaluate the Fine-tuned Model

Check accuracy on the eval set.


In [3]:
eval_result = trainer.evaluate()
print("Evaluation results:")
for k, v in eval_result.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nThe model learned to separate the two classes (accuracy should be high).")


{'eval_loss': '0.6875', 'eval_accuracy': '1', 'eval_runtime': '0.0153', 'eval_samples_per_second': '654.9', 'eval_steps_per_second': '131', 'epoch': '3'}
Evaluation results:
  eval_loss: 0.6875
  eval_accuracy: 1.0000
  eval_runtime: 0.0153
  eval_samples_per_second: 654.8790
  eval_steps_per_second: 130.9760
  epoch: 3.0000

The model learned to separate the two classes (accuracy should be high).


D:\CODE\complete ml\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 9. Freezing Layers

For small datasets, freeze lower layers to prevent overfitting and reduce compute.


In [4]:
# Freeze all layers except the classification head
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params after freezing: {trainable} / {total} ({100*trainable/total:.1f}%)")
print("Only the classification head is trained - this is 'head-only' fine-tuning.")

# Unfreeze for the rest of the notebook
for p in model.parameters():
    p.requires_grad = True


Trainable params after freezing: 66 / 37922 (0.2%)
Only the classification head is trained - this is 'head-only' fine-tuning.


## 10. LoRA Concept (Parameter-Efficient Fine-Tuning)

LoRA trains low-rank adapters instead of full weights. We demonstrate the concept by counting how few parameters a low-rank adapter would add.


In [5]:
# LoRA concept: add a low-rank adapter to a weight matrix
# Instead of updating W (d x d), learn A (d x r) and B (r x d), so W' = W + A@B
d = 32   # hidden size
r = 4    # rank

full_params = d * d
lora_params = d * r + r * d
print(f"Full weight update params: {full_params}")
print(f"LoRA adapter params (rank {r}): {lora_params}")
print(f"Reduction: {100 * (1 - lora_params / full_params):.1f}% fewer params")

# Demonstrate the LoRA forward computation
W = torch.randn(d, d)
A = torch.randn(d, r) * 0.01  # small init
B = torch.randn(r, d)
x = torch.randn(1, d)

y_full = x @ W
y_lora = x @ (W + A @ B)
print("\nLoRA adds a low-rank update A@B to the frozen weight W.")
print("Only A and B are trained, keeping the base model frozen.")


Full weight update params: 1024
LoRA adapter params (rank 4): 256
Reduction: 75.0% fewer params

LoRA adds a low-rank update A@B to the frozen weight W.
Only A and B are trained, keeping the base model frozen.


## 11. Failure Case: Catastrophic Forgetting

Using too high a learning rate destroys pre-trained knowledge.


In [6]:
# Illustrative: high LR vs low LR on the synthetic task
print("With a very high learning rate, the model may overfit the small training set")
print("and forget general patterns (catastrophic forgetting).")
print("\nBest practice: start with 2e-5 to 5e-5 for BERT-like models.")
print("For our tiny model we used 2e-4 because it's much smaller.")


With a very high learning rate, the model may overfit the small training set
and forget general patterns (catastrophic forgetting).

Best practice: start with 2e-5 to 5e-5 for BERT-like models.
For our tiny model we used 2e-4 because it's much smaller.


## 12. Debugging: Common Errors

- **Overfits quickly**: LR too high, data too small. Lower LR, freeze layers.
- **Underfits**: LR too low, model too small. Increase LR, unfreeze layers.
- **Class imbalance**: use weighted loss, oversampling.
- **Fine-tuning degrades performance**: catastrophic forgetting. Use smaller LR, freeze more layers.
- **OOM**: batch size too large. Reduce batch size, use gradient accumulation.

## 13. Real-World Considerations

- Start with 2e-5 to 5e-5 learning rate for BERT-like models.
- Use early stopping based on validation loss.
- For small datasets (<10K), freeze lower layers.
- Use LoRA for large models to reduce compute costs.
- Always evaluate with task-appropriate metrics (not just accuracy).

## 14. Common Mistakes

- Using too high a learning rate.
- Not evaluating during training.
- Not saving checkpoints.
- Ignoring class imbalance.

## 15. When NOT to Use

- When the pretrained model already handles your task well.
- When you have very little labeled data (consider few-shot prompting).

## 16. Challenge

Fine-tune the model with a different number of epochs and compare accuracy.


In [7]:
# Challenge: compare 1 vs 5 epochs
results = {}
for epochs in [1, 5]:
    m = BertForSequenceClassification(BertConfig(**{**config.to_dict(), "num_labels": 2}))
    a = TrainingArguments(
        output_dir=f"./finetune_{epochs}",
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        learning_rate=2e-4,
        report_to=[],
        disable_tqdm=True,
    )
    t = Trainer(model=m, args=a, train_dataset=train_ds, eval_dataset=eval_ds, compute_metrics=compute_metrics)
    t.train()
    results[epochs] = t.evaluate()["eval_accuracy"]
    print(f"{epochs} epoch(s): accuracy={results[epochs]:.3f}")

print("\nMore epochs can help up to a point, then overfitting sets in.")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 47.08it/s]


D:\CODE\complete ml\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'train_runtime': '0.2277', 'train_samples_per_second': '175.7', 'train_steps_per_second': '21.96', 'train_loss': '0.6924', 'epoch': '1'}
{'eval_loss': '0.6916', 'eval_accuracy': '0.5', 'eval_runtime': '0.0171', 'eval_samples_per_second': '584.2', 'eval_steps_per_second': '116.8', 'epoch': '1'}
1 epoch(s): accuracy=0.500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 74.74it/s]

{'train_runtime': '0.6328', 'train_samples_per_second': '316', 'train_steps_per_second': '39.51', 'train_loss': '0.6789', 'epoch': '5'}
{'eval_loss': '0.6713', 'eval_accuracy': '1', 'eval_runtime': '0.0147', 'eval_samples_per_second': '682.5', 'eval_steps_per_second': '136.5', 'epoch': '5'}
5 epoch(s): accuracy=1.000

More epochs can help up to a point, then overfitting sets in.



D:\CODE\complete ml\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 17. Closed-Book Recall

Without looking back:

1. Why should you use a smaller learning rate for fine-tuning?
2. When should you use LoRA instead of full fine-tuning?
3. How do you handle class imbalance during fine-tuning?
4. What metrics should you use beyond accuracy?

## 18. Teach-Back Questions

Explain to another person:

- The fine-tuning workflow.
- The trade-off between full fine-tuning and LoRA.

## 19. Summary

You fine-tuned a tiny transformer for classification, evaluated it, froze layers, and explored the LoRA concept.

## 20. Further Experiment

- Fine-tune a real pretrained model (requires internet).
- Implement LoRA with the `peft` library.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, transformers, datasets, numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
